# PhishGuard: Build Your Own Phishing URL Detector

**BCU Computing Society | Machine Learning for Cybersecurity**  
**Google Colab | Python | Beginner level**

## Your task

Build a small tool that learns from labelled URLs and predicts whether a new URL looks more like a legitimate or phishing example.

You will load a dataset, extract URL features, train a Decision Tree, check its results and test your own examples.

> **Safety:** We only analyse URL text. Do not open URLs from the dataset or enter real suspicious links. A prediction is not proof that a URL is safe or malicious.

## By the end, you will have

- A function that converts a URL into numerical features.
- A trained phishing URL classifier.
- Accuracy, precision, recall and a confusion matrix.
- A function that predicts the category of a new URL.


## A quick introduction to AI and ML

**Artificial Intelligence (AI)** is the wider field of building computer systems that perform tasks associated with intelligent behaviour.

**Machine Learning (ML)** is one approach within AI. We give an algorithm examples, and it learns patterns that can be used to make predictions about new data.

This workshop uses a **classifier**. It does not generate text like a chatbot. It receives numerical information about a URL and predicts a category.

### How it works

```text
Labelled URLs -> Training -> Trained model
                                  |
New URL -> Extract features -> Prediction
```

| Term | In this project |
|---|---|
| Dataset | URLs with known labels |
| Feature | A measurable URL characteristic, such as its length |
| Label | Legitimate or phishing |
| Training | Learning from labelled examples |
| Model | The patterns learned by the algorithm |
| Prediction | The category assigned to a new URL |

The model learns from the examples and labels. It does not know the intent of the person who created a URL.


## What are we building?

Some URLs contain characteristics that may be useful signals, such as a very long address, an IP address in place of a domain, or several subdomains. These clues can also appear in legitimate URLs, so no single feature proves that a URL is malicious.

Our workflow:

1. Load labelled URLs.
2. Extract numerical features from each URL string.
3. Split the examples into training and testing sets.
4. Train a Decision Tree classifier.
5. Evaluate the model and inspect its mistakes.
6. Test new fictional URLs.

**We will not visit websites, download files, inspect webpage content or connect this model to a live service.**


## How to use this notebook

Run the cells from top to bottom. In Colab, select **Runtime -> Run all**, or use each cell's play button. Read the short notes, run the code and check the output before continuing.

You do not need to write the whole program yourself. The final sections let you change a few values and experiment.


## 1. Set up the environment

We use pandas for tables, NumPy for data operations, Matplotlib for charts and scikit-learn to train and evaluate the model. The `ucimlrepo` package loads the dataset from UCI.

Run the cell to install and import the tools. Colab runs Python online, so no local setup is needed.


In [ ]:
%pip -q install ucimlrepo

# Standard library tools for parsing URL text and handling warnings.
import re
import ipaddress
import warnings
from urllib.parse import urlsplit

# Data handling and charts.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display tables neatly inside Colab.
from IPython.display import display

# Download the labelled dataset from the UCI repository.
from ucimlrepo import fetch_ucirepo

# Tools for splitting data, training the tree and measuring its results.
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
)

# Hide a few harmless display warnings so they do not distract from the lesson.
warnings.filterwarnings("ignore", category=UserWarning)

# Keep random operations repeatable for students using the same data and settings.
RANDOM_STATE = 42
print("Environment ready. Let's load the dataset.")

## 2. Load the dataset

We use the **PhiUSIIL Phishing URL Dataset** from the UCI Machine Learning Repository. It contains 235,795 labelled examples.

**Check the original labels:** UCI uses `1 = legitimate` and `0 = phishing`. Later, we convert them to `0 = legitimate` and `1 = phishing`, so phishing is the positive class in our evaluation.

Run the cell to download the dataset. The file is around 54 MB, so this step may take a little while.


In [ ]:
# Retrieve dataset 967 from the UCI Machine Learning Repository.
# This downloads the dataset file; it does not open or visit the URLs in it.
phiusiil = fetch_ucirepo(id=967)

# Copy the URL information and the labels into separate tables.
raw_features = phiusiil.data.features.copy()
raw_targets = phiusiil.data.targets.copy()

# Show the dataset dimensions and column names so we can inspect its structure.
print("Dataset downloaded successfully.")
print("Feature table shape:", raw_features.shape)
print("Target table shape:", raw_targets.shape)
print("Feature columns include:", list(raw_features.columns[:12]))
print("Target columns:", list(raw_targets.columns))

### If the download fails

Download the CSV from the [official UCI dataset page](https://archive.ics.uci.edu/dataset/967/phiusil+phishing+url+dataset), extract it if needed, then run the optional upload cell below and choose `PhiUSIIL_Phishing_URL_Dataset.csv`.

Only download the dataset itself. Do not open any URLs stored inside it.


In [ ]:
# OPTIONAL FALLBACK: run this cell only if the automatic UCI download failed.
# If the download worked, the variables already exist and the upload is skipped.
if "raw_features" in globals() and "raw_targets" in globals():
    print("Automatic dataset download already succeeded. Skipping manual upload.")
else:
    # Colab's upload window lets you select the CSV downloaded from the UCI page.
    from google.colab import files
    uploaded = files.upload()

    # Keep only uploaded files whose names end in .csv.
    csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
    if not csv_files:
        raise ValueError("No CSV file was uploaded. Please upload the official PhiUSIIL CSV.")

    # Read the selected CSV into a pandas DataFrame.
    uploaded_df = pd.read_csv(csv_files[0], low_memory=False)

    # Find the URL and label columns without depending on upper/lower case.
    url_column_fallback = next((c for c in uploaded_df.columns if c.lower() == "url"), None)
    label_column_fallback = next((c for c in uploaded_df.columns if c.lower() == "label"), None)

    # Stop with a clear message if the expected columns are missing.
    if url_column_fallback is None or label_column_fallback is None:
        raise ValueError("The uploaded CSV must contain URL and label columns.")

    # Keep only the two columns needed for this workshop.
    raw_features = uploaded_df[[url_column_fallback]].copy()
    raw_targets = uploaded_df[[label_column_fallback]].copy()

    print("Uploaded dataset loaded:", uploaded_df.shape)

## 3. Inspect the dataset

Each row contains a URL and its published label. The original dataset also includes webpage-derived information. We ignore those columns and calculate our own features from the URL text, so the same function works on a new URL without visiting it.


In [ ]:
# Find the URL and label columns, ignoring differences in capitalisation.
url_column = next((c for c in raw_features.columns if c.lower() == "url"), None)
label_column = next((c for c in raw_targets.columns if c.lower() == "label"), None)

# Check that both columns were found before trying to use them.
if url_column is None:
    raise ValueError("Could not find a URL column in the UCI feature table.")
if label_column is None:
    # Some dataset interfaces return a single target column with a different name.
    if raw_targets.shape[1] == 1:
        label_column = raw_targets.columns[0]
    else:
        raise ValueError("Could not identify the label column in the target table.")

# Convert URLs to text, remove surrounding spaces and convert labels to numbers.
url_series = raw_features[url_column].astype("string").str.strip()
original_labels = pd.to_numeric(raw_targets[label_column], errors="coerce")

# Keep rows with a non-empty URL and a valid original label (0 or 1).
valid_rows = url_series.notna() & url_series.ne("") & original_labels.isin([0, 1])

# Build a clean table containing only the URL and its original label.
data = pd.DataFrame({
    "url": url_series[valid_rows].astype(str).values,
    "original_label": original_labels[valid_rows].astype(int).values,
})

# UCI labels: 1 = legitimate and 0 = phishing.
# Workshop labels: 0 = legitimate and 1 = phishing.
# This keeps phishing as the positive class for precision and recall.
data["label"] = (data["original_label"] == 0).astype(int)

# Remove duplicate URLs and keep only the fields needed for modelling.
data = data[["url", "label"]].drop_duplicates(subset="url").reset_index(drop=True)

# Show how many examples remain in each class after cleaning.
print("Usable unique URLs:", f"{len(data):,}")
print("\nWorkshop label counts:")
print(data["label"].value_counts().rename(index={0: "Legitimate (0)", 1: "Phishing (1)"}))
print("\nFirst five examples:")
display(data.head())

### Quick check

- Why do we need known labels when training this model?
- Could a dataset label be incorrect or out of date?

The model learns from the labels it is given. Poor labels can produce a poor model.


In [ ]:
# Count how many URLs belong to each class (0 = legitimate, 1 = phishing).
label_counts = data["label"].value_counts().sort_index()

# Replace numeric labels with readable names for the chart.
label_counts.index = ["Legitimate", "Phishing"]

# Draw one bar for each class so we can see the dataset balance.
ax = label_counts.plot(kind="bar", figsize=(7, 4), rot=0)
ax.set_title("URLs in the dataset")
ax.set_xlabel("Dataset label")
ax.set_ylabel("Number of examples")
plt.tight_layout()  # Fit labels inside the figure.
plt.show()          # Display the chart in Colab.

## 4. Understand the URL features

A **feature** is a piece of information the model uses to make a prediction. We turn each URL into a row of numbers.

| Feature | What it measures |
|---|---|
| `url_length` | Total URL characters |
| `hostname_length` | Hostname characters |
| `dots_in_hostname` | Dots in the hostname |
| `hyphens_in_hostname` | Hyphens in the hostname |
| `digits_in_url` | Numeric characters in the URL |
| `special_characters` | Selected characters such as `@`, `?`, `=`, `&`, `%` |
| `path_length` | Length of the path |
| `query_length` | Length of the query string |
| `has_ip_address` | 1 if the hostname is an IP address, otherwise 0 |
| `uses_https` | 1 if the scheme is HTTPS, otherwise 0 |
| `has_at_symbol` | 1 if the URL contains `@`, otherwise 0 |
| `keyword_count` | Count of selected words such as `login` and `verify` |

These are simple signals, not rules. HTTPS, digits and long URLs can all be legitimate.


### Parts of a URL

```text
https://shop.example.com/products?id=42
       |               |        |
    hostname          path    query
```

The function below reads these parts as text. It does not connect to the website.


In [ ]:
# Use a reserved example domain so no real website is involved.
example_url = "http://account-check.example/login/verify?id=12345"

# Split the URL into its parts without making a network request.
parsed_example = urlsplit(example_url)

# Print the components so we can see what the parser extracted.
print("Full URL:", example_url)
print("Scheme:", parsed_example.scheme)
print("Hostname:", parsed_example.hostname)
print("Path:", parsed_example.path)
print("Query:", parsed_example.query)

## 5. Create the feature extractor

`extract_url_features()` takes a URL string and returns the same 12 numerical features every time. We use this function for both training data and new URLs, so the model receives information in a consistent format.


In [ ]:
# Words that we will count if they appear in the URL text.
KEYWORDS = ["login", "signin", "verify", "account", "update", "secure", "password", "bank"]

# Keep the feature names in one place so training and prediction use the same order.
FEATURE_COLUMNS = [
    "url_length",
    "hostname_length",
    "dots_in_hostname",
    "hyphens_in_hostname",
    "digits_in_url",
    "special_characters",
    "path_length",
    "query_length",
    "has_ip_address",
    "uses_https",
    "has_at_symbol",
    "keyword_count",
]


def extract_url_features(url):
    """Turn one URL string into numerical features. The function never visits it."""
    # Convert the input to text and remove spaces at the beginning or end.
    original = str(url).strip()

    # urlsplit needs a scheme (https://) or a // prefix to find the hostname.
    # Add // temporarily when the user supplies a URL without a scheme.
    candidate = original
    if not re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", candidate):
        candidate = "//" + candidate

    # Separate the hostname, path and query string from the URL text.
    parsed = urlsplit(candidate)
    hostname = (parsed.hostname or "").lower()
    path = parsed.path or ""
    query = parsed.query or ""

    # Check whether the hostname is an IPv4 or IPv6 address.
    try:
        ipaddress.ip_address(hostname)
        has_ip_address = 1  # The hostname is an IP address.
    except ValueError:
        has_ip_address = 0  # The hostname is not a recognised IP address.

    # Count selected words found anywhere in the lower-case URL.
    lower_url = original.lower()
    keyword_count = sum(1 for word in KEYWORDS if word in lower_url)

    # Return one number for every feature listed in FEATURE_COLUMNS.
    return {
        "url_length": len(original),  # Total number of characters.
        "hostname_length": len(hostname),  # Characters in the hostname.
        "dots_in_hostname": hostname.count("."),  # Number of dots in the hostname.
        "hyphens_in_hostname": hostname.count("-"),  # Number of hyphens in the hostname.
        "digits_in_url": sum(char.isdigit() for char in original),  # Count numeric characters.
        "special_characters": sum(original.count(char) for char in "@?=&%_"),  # Count selected symbols.
        "path_length": len(path),  # Number of characters in the path.
        "query_length": len(query),  # Number of characters after the ? in the query.
        "has_ip_address": has_ip_address,  # 1 for an IP hostname, otherwise 0.
        "uses_https": int(parsed.scheme.lower() == "https"),  # 1 for HTTPS, otherwise 0.
        "has_at_symbol": int("@" in original),  # 1 if @ appears in the URL.
        "keyword_count": keyword_count,  # Number of selected keywords found.
    }


# Try the function on the fictional URL above and display the resulting numbers.
example_features = extract_url_features(example_url)
display(pd.DataFrame([example_features]))

### Check your understanding

1. Which feature changes if you add a long path?
2. Which feature changes if you replace the hostname with an IP address?
3. Does HTTPS guarantee that a website is legitimate?


## 6. Select a smaller sample

The full dataset is larger than we need for this workshop. We select up to **30,000 examples**, keeping a similar balance of legitimate and phishing URLs. This makes feature extraction and training faster in Colab.


In [ ]:
# Set the maximum number of URL examples used in the workshop.
MAX_EXAMPLES = 30_000

if len(data) > MAX_EXAMPLES:
    # Select a smaller sample while keeping the same label proportions.
    selected_indices, _ = train_test_split(
        np.arange(len(data)),                 # Numeric position of every row.
        train_size=MAX_EXAMPLES,              # Number of rows to keep.
        random_state=RANDOM_STATE,            # Make the selection repeatable.
        stratify=data["label"],               # Preserve legitimate/phishing balance.
    )
    # Use the selected rows and reset their row numbers.
    workshop_data = data.iloc[selected_indices].reset_index(drop=True)
else:
    # If the dataset is already small, use all available rows.
    workshop_data = data.copy().reset_index(drop=True)

# Check the sample size and number of examples in each class.
print("Examples selected for this workshop:", f"{len(workshop_data):,}")
print("Legitimate:", int((workshop_data["label"] == 0).sum()))
print("Phishing:", int((workshop_data["label"] == 1).sum()))

## 7. Convert URLs into features

We apply the feature extractor to every selected URL. `X` will contain the numerical features, and `y` will contain the known labels.


In [ ]:
# Run the feature extractor for every URL and store each result as a dictionary.
feature_records = [extract_url_features(url) for url in workshop_data["url"]]

# Convert the dictionaries into a table: rows are URLs and columns are features.
X = pd.DataFrame(feature_records, columns=FEATURE_COLUMNS)

# Store the correct answers separately. 0 = legitimate, 1 = phishing.
y = workshop_data["label"].astype(int).reset_index(drop=True)

# Show the shapes to confirm that X and y have the same number of examples.
print("Feature table shape:", X.shape)
print("Label vector shape:", y.shape)
print("\nFirst five feature rows:")
display(X.head())
print("\nFirst five labels (0 = Legitimate, 1 = Phishing):")
print(y.head().to_list())

### What are X and y?

- **`X`** contains the input features used to make predictions.
- **`y`** contains the correct labels the model learns to predict.

This is **supervised learning** because each training example has a known label.


In [ ]:
# X contains the information given to the model; y contains the answers.
print("Feature names:")
print(FEATURE_COLUMNS)
print("\nLabel meaning:")
print({0: "Legitimate", 1: "Phishing"})

## 8. Split training and testing data

We use **80% for training** and **20% for testing**. The test examples are held back so we can check how the model performs on data it did not train on.

`stratify=y` keeps a similar balance of labels in both groups. `random_state=42` makes the split repeatable.


In [ ]:
# Split X and y together so each feature row stays matched to its label.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,          # Keep 20% aside for the final test.
    random_state=RANDOM_STATE,  # Use the same split each time.
    stratify=y,               # Keep a similar class balance in both sets.
)

# Show the number of examples assigned to each group.
print("Training examples:", f"{len(X_train):,}")
print("Testing examples:", f"{len(X_test):,}")

# Display the class proportions to check that stratification worked.
print("\nTraining label balance:")
print(y_train.value_counts(normalize=True).rename(index={0: "Legitimate", 1: "Phishing"}).round(3))
print("\nTesting label balance:")
print(y_test.value_counts(normalize=True).rename(index={0: "Legitimate", 1: "Phishing"}).round(3))

### Why keep test data separate?

A model could memorise details in its training examples. Testing it on unseen examples helps us see whether it has learned patterns that generalise. Poor performance on unseen data can be a sign of **overfitting**.


## 9. Train the Decision Tree

A Decision Tree makes a prediction by following questions about the features. The algorithm chooses useful splits from the training examples.

We limit the tree's depth and the minimum number of examples in each leaf to keep it manageable and reduce overfitting.


In [ ]:
# Create the Decision Tree model and set limits on its complexity.
model = DecisionTreeClassifier(
    max_depth=6,             # The tree can ask up to six levels of questions.
    min_samples_leaf=25,     # Each final leaf must contain at least 25 examples.
    class_weight="balanced",  # Give extra weight to the less common class if needed.
    random_state=RANDOM_STATE,  # Keep training repeatable.
)

# Fit means: let the algorithm learn patterns from X_train and y_train.
model.fit(X_train, y_train)
print("PhishGuard model trained successfully.")

## 10. Evaluate the model

We compare predictions with the correct labels in the test set:

- **Accuracy:** proportion of all predictions that are correct.
- **Precision:** when the model flags phishing, how often is it correct?
- **Recall:** how many of the phishing examples did it identify?
- **Confusion matrix:** shows correct predictions and mistakes.

Accuracy alone is not enough. In cybersecurity, missed phishing URLs and legitimate URLs incorrectly flagged can both cause problems.


In [ ]:
# Ask the trained model to predict labels for URLs it did not train on.
y_pred = model.predict(X_test)

# Compare the predictions with the correct test labels.
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

# Print the three scores. Precision and recall focus on the phishing class (1).
print(f"Accuracy:  {accuracy:.3f} ({accuracy * 100:.1f}%)")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")

# Show precision, recall and F1-score separately for both label categories.
print("\nFull classification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],  # Keep the report order consistent with our label convention.
    target_names=["Legitimate", "Phishing"],
    zero_division=0,  # Avoid a divide-by-zero warning if a class is never predicted.
))

### Understand the confusion matrix

- **True negative:** legitimate URL correctly identified.
- **False positive:** legitimate URL incorrectly flagged as phishing.
- **False negative:** phishing URL missed by the model.
- **True positive:** phishing URL correctly identified.

False positives can waste analyst time. False negatives mean suspicious URLs may be missed.


In [ ]:
# Compare the correct labels with the model predictions.
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

# Give the matrix readable labels and display it as a chart.
display_cm = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Legitimate", "Phishing"],
)
display_cm.plot(cmap="Blues", values_format="d")
plt.title("PhishGuard: Confusion Matrix")
plt.tight_layout()
plt.show()

# Unpack the four cells in the order used by scikit-learn:
# true negative, false positive, false negative, true positive.
true_negative, false_positive, false_negative, true_positive = cm.ravel()
print("True negatives:", true_negative)
print("False positives:", false_positive)
print("False negatives:", false_negative)
print("True positives:", true_positive)

### Discuss the result

1. How many legitimate URLs were flagged incorrectly?
2. How many phishing URLs were missed?
3. Would you block a URL based only on this prediction? Why?

The result applies to this dataset, feature set, model and test split. It does not guarantee performance on current URLs.


## 11. Visualise the Decision Tree

The diagram shows some of the questions the model learned. Only the first few levels are displayed so the tree is easier to read.


In [ ]:
# Start a large figure so the tree labels are easier to read.
plt.figure(figsize=(22, 10))

# Draw only the first three levels. The trained model itself is not shortened.
plot_tree(
    model,
    feature_names=FEATURE_COLUMNS,  # Show meaningful feature names at each split.
    class_names=["Legitimate", "Phishing"],  # Show readable class names in leaves.
    filled=True,                  # Colour nodes by their predicted class.
    rounded=True,                 # Use rounded boxes for easier reading.
    max_depth=3,                  # Limit only the visual display depth.
    fontsize=8,
)
plt.title("A simplified view of the PhishGuard Decision Tree")
plt.tight_layout()
plt.show()

## 12. Feature importance

Feature importance shows which features the fitted tree relied on most. The ranking can change when the data or model settings change. It does not show that a feature causes phishing.


In [ ]:
# Pair each feature name with the importance calculated by the fitted tree.
importance = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)  # Put the most-used features first.

# Display the values rounded to four decimal places.
display(importance.round(4))

# Draw a horizontal bar chart to compare feature importance.
plt.figure(figsize=(9, 5))
plt.barh(importance["feature"], importance["importance"])
plt.gca().invert_yaxis()  # Put the most important feature at the top.
plt.xlabel("Feature importance")
plt.title("Which URL features did this Decision Tree use?")
plt.tight_layout()
plt.show()

## 13. Build the URL checker

`check_url()` extracts the same features used during training, passes them to the model and prints its prediction. It does not open the URL or check its reputation.


In [ ]:
def check_url(url, show_features=False):
    """Predict a URL category from its text. The URL is never visited."""
    # Convert the URL into the same feature values used during model training.
    features = extract_url_features(url)

    # A DataFrame keeps the feature names and column order expected by the model.
    input_row = pd.DataFrame([features], columns=FEATURE_COLUMNS)

    # Predict the label: 0 = legitimate, 1 = phishing.
    prediction = int(model.predict(input_row)[0])

    # Turn the numeric prediction into a message students can understand.
    if prediction == 1:
        result = "POTENTIALLY PHISHING"
        explanation = "The URL has patterns associated with phishing examples in the training data."
    else:
        result = "LEGITIMATE-LIKE"
        explanation = "The URL has patterns associated with legitimate examples in the training data."

    # Print the result and remind the user that it is not a security verdict.
    print("URL:", url)
    print("Model prediction:", result)
    print(explanation)
    print("\nThis is a model prediction, not proof that the URL is safe or malicious.")

    # Show the numerical features if the caller asks for them.
    if show_features:
        print("\nExtracted features:")
        display(input_row)

    # Return the numeric class in case another part of the notebook needs it.
    return prediction

print("URL checker ready.")

### Test A: fictional organisation URL

This uses the reserved `.example` domain. The model's result depends on what it learned during training.


In [ ]:
# Pass a fictional URL to the checker and display the features it extracts.
example_legitimate = "https://portal.example.edu/student/home"
result_a = check_url(example_legitimate, show_features=True)

### Test B: fictional login-themed URL

This is also a reserved example address. Compare the result with Test A. Do not open it.


In [ ]:
# This fictional URL contains more login-related words and URL symbols.
example_suspicious = "http://account-verify-login.example/signin?user=12345&session=98765"
result_b = check_url(example_suspicious, show_features=True)

### Test C: try your own URL

Change the fictional `.example` URL in the next cell. Try changing one thing at a time, such as the path length, number of digits, subdomains or use of HTTPS. Watch how the extracted features and prediction change.


In [ ]:
# Change this example to test another fictional .example URL.
my_test_url = "https://services.example.com/account/home"
my_prediction = check_url(my_test_url, show_features=True)

## 14. Challenge: can you confuse the model?

Create two fictional URLs:

1. One that looks unusual to you but is predicted as legitimate-like.
2. One that looks ordinary to you but is predicted as potentially phishing.

Use `.example` domains only. Do not visit the URLs.

Discuss: which features influenced the prediction, and what other information would help an analyst decide whether to trust the URL?


In [ ]:
# CHALLENGE: edit the two fictional URLs, then run the cell again.
challenge_url_1 = "http://very-long-account-verification.example/login?ref=123456789"
challenge_url_2 = "https://news.example.org/article/2026/security-update"

# Ask the model to classify each URL and print its result.
print("Challenge example 1")
challenge_prediction_1 = check_url(challenge_url_1)

print("\nChallenge example 2")
challenge_prediction_2 = check_url(challenge_url_2)

## 15. Optional experiment: change the tree depth

Try different values for `max_depth`. A deeper tree can learn more detailed patterns, but it can also overfit. Compare training accuracy with testing accuracy. The highest score is not automatically the best model.


In [ ]:
# Store the results so we can compare each tree depth after the loop.
depth_results = []

# Train a separate model with each depth setting.
for depth in [2, 4, 6, 10, 14]:
    experiment_model = DecisionTreeClassifier(
        max_depth=depth,          # Try a shallower or deeper tree.
        min_samples_leaf=25,      # Keep the minimum leaf size fixed.
        class_weight="balanced", # Keep class weighting consistent.
        random_state=RANDOM_STATE,
    )

    # Train this version using the same training data.
    experiment_model.fit(X_train, y_train)

    # Measure performance on training data and on held-back test data.
    train_accuracy = experiment_model.score(X_train, y_train)
    test_accuracy = experiment_model.score(X_test, y_test)

    # Save the depth and both scores as one row for the results table.
    depth_results.append({
        "max_depth": depth,
        "training_accuracy": train_accuracy,
        "testing_accuracy": test_accuracy,
    })

# Convert the saved results into a table and display rounded scores.
depth_results_df = pd.DataFrame(depth_results)
display(depth_results_df.round(3))

# Plot training and testing accuracy to compare the depth settings.
plt.figure(figsize=(8, 4))
plt.plot(depth_results_df["max_depth"], depth_results_df["training_accuracy"], marker="o", label="Training accuracy")
plt.plot(depth_results_df["max_depth"], depth_results_df["testing_accuracy"], marker="o", label="Testing accuracy")
plt.xlabel("Maximum tree depth")
plt.ylabel("Accuracy")
plt.title("How tree depth affects model performance")
plt.legend()
plt.tight_layout()
plt.show()

## Using an AI assistant

You can ask an AI assistant to explain a code cell or help interpret an error. The **Decision Tree**, not the chatbot, makes the phishing prediction. Check suggested code before running it, and never share private URLs or personal information.


## 16. Limitations and safe use

PhishGuard is a learning prototype, not a production phishing detector.

- It uses URL text only. It does not inspect the website, redirects, certificates or domain reputation.
- URL features are clues, not proof. Legitimate sites can have the same characteristics as phishing sites.
- The model learns from the dataset's labels, which may be incomplete or out of date.
- A random test split does not prove that the model will work on new phishing campaigns or domains.

**Safety:** use fictional `.example` URLs for testing. Do not open dataset URLs, enter live suspicious links or automatically block real traffic based on this model.


## 17. What you learned

You have used a labelled dataset to train a Decision Tree classifier.

You have practised:

- Turning URL text into numerical features.
- Separating training data from test data.
- Training and evaluating a classifier.
- Reading a confusion matrix and finding model errors.
- Using the trained model to predict a new example.

A prediction is a useful signal for further investigation, not proof that a URL is safe or malicious.


## Dataset acknowledgement and further learning

**Dataset:** Prasad, A. & Chandra, S. (2024). *PhiUSIIL Phishing URL (Website)*. UCI Machine Learning Repository. [Dataset page](https://archive.ics.uci.edu/dataset/967/phiusil+phishing+url+dataset). Licensed under CC BY 4.0. Keep this acknowledgement if you reuse the dataset.

**Further learning:**

- [Google Teachable Machine](https://teachablemachine.withgoogle.com/)
- [scikit-learn: Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- [scikit-learn: Model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [UCI Machine Learning Repository](https://archive.ics.uci.edu/)

**BCU Computing Society | PhishGuard: Build Your Own Phishing URL Detector**
